# core

> Scheme implementation in Python

Goal here is to incrementally build up to an implementation of Lisp in python. I'd like to use the solveit method starting from the inside out. So we'll likely start with literals and work our way up.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great. An inside-out path for a tiny Lisp in Python could start like this:

1. **Data/literals**: numbers, strings, booleans, symbols, lists
2. **Reader/parser**: turn text like `(+ 1 2)` into Python data structures
3. **Environment**: variable lookup and bindings
4. **Evaluator**: evaluate literals, symbols, and function calls
5. **Special forms**: `quote`, `if`, `define`, `lambda`
6. **Functions/macros later**: builtins, closures, maybe macros

For the very first step, I’d separate **self-evaluating literals** from **symbols**.

A minimal representation might be:

```python
class Symbol(str): pass
```

Then plain Python values can represent literals:

```python
42          # number literal
"hello"     # string literal
True        # boolean literal
Symbol("x") # Lisp symbol
```

The key distinction: `"x"` is a string value, but `Symbol("x")` means “look up the variable named x” once evaluation exists.

<details class='token-usage-details' markdown='1'><summary>$0.0048</summary>

`total=30,987 | in=30,751 | out=236 | cached=4.6% | $0.0048 | gpt-5.5`

</details>


In [ ]:
#| default_exp core

In [ ]:
#| export
import math, operator as op

In [ ]:
#| export
from fastcore.basics import store_attr, first

In [ ]:
#| export
from compact.reader import *
from compact.types import *

help me understand the purpose of env in Lisp

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

`env` is the interpreter’s **memory of names**.

When Lisp sees a symbol like:

```scheme
x
```

it does not know by itself what `x` means. The environment tells it:

```python
env["x"] == 10
```

So if the evaluator sees `x`, it looks up `x` in `env` and gets `10`.

For example, after something like:

```scheme
(define radius 5)
```

the environment records:

- symbol/name: `radius`
- value: `5`

Then later:

```scheme
(* pi radius radius)
```

can be evaluated because `env` knows what `pi`, `radius`, and `*` mean.

A useful mental model:

- **expression** = the thing we are evaluating
- **env** = the context that gives names meaning

Without `env`, the interpreter could evaluate literal numbers like `3`, but it could not understand variables, built-in functions, or user-defined functions.

<details class='token-usage-details' markdown='1'><summary>$0.0050</summary>

`total=31,889 | in=31,585 | out=304 | cached=4.5% | reasoning=91 | $0.0050 | gpt-5.5`

</details>


### Environment

In [ ]:
#| export
class Env:
    "implementation of env for scheme eval()."
    def __init__(self, d, primitives:dict=(), parent=None): store_attr(cast=True)

    def __getitem__(self, k):
        s = self.find(k)
        if s is not None: 
            # there could be collisions for keys in primitives and globals()
            # for instance if `list` is called in scheme we want the builtin version
            # in python list will return what is in globals
            # (define list 42) is the moral equivalent of list = 42 - they will overwrite what's in globals
            if k in s.primitives: return s.primitives[k]
            return s.d[k]
        raise KeyError(k)

    def __setitem__(self, k, v): self.d[k] = v

    def __contains__(self, k): return self.find(k) is not None

    def update(self, d): self.primitives.update(d)

    def find(self, k):
        if k in self.d | self.primitives: return self
        if self.parent is not None: return self.parent.find(k)
        return None
    
    def is_primitive(self, k):
        if self.parent is None: return k in self.primitives
        return self.parent.is_primitive(k)

    def new_frame(self, bindings=()):
        return Env(dict(bindings), parent=self)


### The Evaluator: eval and apply

help me understand how eval and apply work together?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

`eval` and `apply` are the two halves of a Lisp interpreter:

- **`eval`** answers: “What does this expression mean in this environment?”
- **`apply`** answers: “Given a function and already-evaluated arguments, how do I call it?”

For a call expression like:

```scheme
(+ 1 2)
```

`eval` does the outer work:

1. sees this is a list
2. evaluates the operator `+` → finds the Python/Lisp function for addition
3. evaluates each argument: `1 → 1`, `2 → 2`
4. hands them to `apply`

Conceptually:

```python
proc = eval(op, env)
args = [eval(arg, env) for arg in expr[1:]]
return apply(proc, args)
```

Then `apply` actually invokes the procedure.

The important split is: **`eval` decides what must be evaluated; `apply` just calls something with values**. Special forms like `if` and `define` are special because they *don’t* evaluate all their arguments before acting.

<details class='token-usage-details' markdown='1'><summary>$0.0049</summary>

`total=31,455 | in=31,219 | out=236 | cached=20.9% | $0.0049 | gpt-5.5`

</details>


In [ ]:
#| export
def scm_apply(fn, args): return fn(*args)

In [ ]:
#| export
def scm_eval_one_step(expr, env, sfs=()):
    "eval-uate lisp expressions given an environment and handlers for special forms"
    # Atoms
    if isinstance(expr, Symbol): return env[expr.s]
    if not isinstance(expr, list): return expr

    # lists
    hd, body = expr[0], expr[1:]

    if isinstance(hd, Symbol) and hd.s in sfs: return sfs[hd.s](body, env, sfs)
    fn = scm_eval_tco(hd, env, sfs)
    if isinstance(fn, Macro): return Thunk(fn.fn(*body), env)

    return scm_apply(fn, [scm_eval_tco(o, env, sfs) for o in body])

In [ ]:
#| export
def scm_eval_tco(expr, env, sfs=()):
    "eval but with tail call optimization"
    while True:
        r = scm_eval_one_step(expr, env, sfs)
        if isinstance(r, Thunk): expr, env = r.expr, r.env
        else: return r

**Tail-Call Optimization**
Naive recursive evaluation stack overflows for deep recursion (e.g. `(fact 10000)`). Instead of recursing, we can evaluate iteratively.`scm_eval_one_step` and `scm_eval_tco` recursively call each other.  

One step returns expression, if it is a `Thunk`, it gets executed in the scm_eval_tco. This avoids having deep stacks.

### Primitive Procedures
Give the interpreter some basic skills.

In [ ]:
#| export
def scm_sub(x, *xs): return x - sum(xs) if xs else -x
def scm_div(x, *xs): return 1/x if not xs else x / math.prod(xs)

def is_num(x): return not isinstance(x, bool) and isinstance(x, (int, float, complex))
def is_sym(x): return isinstance(x, Symbol)

_builtin = {
    "+": lambda *xs: sum(xs),
    "*": lambda *xs: math.prod(xs), 
    "/": scm_div,
    "-": scm_sub,
    ">": op.gt, 
    "<": op.lt, 
    "=": lambda x,y: is_num(x) and is_num(y) and x == y,
    "<=": op.le,
    ">=": op.ge,
    
    "number?": is_num,
    "string?": lambda x: isinstance(x, str),
    "symbol?": is_sym,
    "boolean?": lambda x: isinstance(x, bool),
}

def is_list(x): return isinstance(x, list)
def is_pair(x): return isinstance(x, list) and bool(x)

_builtin |= {
    "list": lambda *xs: list(xs),
    "cons": lambda x,y: [x] + y,
    "car": lambda x: x[0],
    "cdr": lambda x: x[1:],
    "null?": lambda x: x == [],
    "pair?": is_pair,
    "list?": is_list,
}

In [ ]:
global_env = Env(globals(), parent=None, primitives=_builtin)

We want our tiny lisp implemention to interop with symbols in python so the top level "frame" is globals()

### Special Forms
eval'ing a list will cause all of the arguments of the procedure to be evaluated before the procedure is called.

For instance:
```
    (+ 2 (* 4 5))
```

before plus (+) is called, all arguments are evaluated and this becomes `(+ 2 20)`

Special forms DO NOT evaluate the arguments. It is up to the the special form to decide when and how evaluate/interpret them.

For instance:
```
(if #t 1 (/ 1 0))
```
The `else` clause should not be greedily evaluated. `if` needs to figure out if it wants to handle it.

#### quote
```lisp
    `(+ 1 2) -> (+ 1 2)
```

In [ ]:
#| export
def _sf_quote(xs, env, sfs): return xs[0]

#### begin

```lisp
    (begin
        (define a 2)
        (+ a 2))
```

In [ ]:
#| export
def _sf_begin_tco(xs, env, sfs):
    for o in xs[:-1]: scm_eval_tco(o, env, sfs)
    return Thunk(xs[-1], env)

In [ ]:
#| export
def body_expr(body): return body[0] if len(body) == 1 else [Symbol("begin")] + body

Convenience function to handle multi expression lists

#### if

```lisp
    (if #t 1 (/ 1 0))
```

In [ ]:
#| export
def _sf_if_tco(xs, env, sfs):
    cond,then_,else_ = xs
    br = else_ if scm_eval_tco(cond, env, sfs) is False else then_
    return Thunk(br, env)


#### lambda — anonymous procedures

```lisp
    (lambda (x) (+ x 2))
```

In [ ]:
#| export
def _sf_lambda_tco(xs, env, sfs): 
    params, *body = xs
    def procedure(*args):
        return Thunk(body_expr(body), env.new_frame(bind_params(params, args)))
    return procedure
    # return lambda *args: Thunk(body_expr(body), Env(**bind_params(params, args), parent=env))

#### define — binding symbols

```lisp
    (define name value)

    (define square (lambda (x) (* x x)))

    (define (square x) (* x x))
```

In [ ]:
#| export
def arity(ps, vs):
    if len(vs) == len(ps): return vs
    raise ValueError(f"incorrect number of vs passed (wanted: {len(ps)}, got: {len(vs)})")

def bind_params(ps, vs):
    if isinstance(ps, Symbol): return {ps.s: list(vs)}
    return dict(zip([p.s for p in ps], arity(ps, vs)))

In [ ]:
#| export
def _sf_define_tco(xs, env, sfs):
    def _mkfn(sym, expr): 
        if not isinstance(sym, Symbol): raise SyntaxError(f"{sym} must be a symbol")
        if env.is_primitive(sym.s): raise SyntaxError(f"cannot redefine primitive '{sym.s}'")
        env[sym.s] = scm_eval_tco(expr, env, sfs)
        return sym.s

    arg0, *rest = xs
    if isinstance(arg0, Symbol): return _mkfn(arg0, body_expr(rest))
    if isinstance(arg0, list): return _mkfn(arg0[0], [Symbol("lambda"), arg0[1:]] + rest)

#### set!

In [ ]:
#| export
def _sf_set(xs, env, sfs):
    sym, expr = xs
    if not isinstance(sym, Symbol): raise SyntaxError(f"set! argument {sym} must be a symbol")
    e = env.find(sym.s)
    if e is None: raise NameError(sym.s)

    e[sym.s] = scm_eval_tco(expr, env, sfs)
    return sym.s

In [ ]:
scm_eval_tco(parse("""
(begin
  (define x 1)
  (set! x 42)
  x)
"""), global_env, sfs={"begin": _sf_begin_tco, "define": _sf_define_tco, "set!": _sf_set})

42

In [ ]:
scm_eval_tco(
    parse("""
        (begin
          (define fact-iter
            (lambda (n acc)
            (if (= n 0) acc (fact-iter (- n 1) (* acc n)))))
          (fact-iter 5 1))
    """), 
    global_env, 
    sfs={"define":_sf_define_tco, "lambda":_sf_lambda_tco, "if":_sf_if_tco, "begin":_sf_begin_tco}
)

120

In [ ]:
scm_eval_tco(
    parse("""

(begin
  (define fact-iter
    (lambda (n acc)
    (if (= n 0) acc (fact-iter (- n 1) (* acc n)))))
  (fact-iter 1000 1))        
  
    """), 
    global_env, 
    sfs={"define":_sf_define_tco, "lambda":_sf_lambda_tco, "if":_sf_if_tco, "begin":_sf_begin_tco}
)

4023872600770937735437024339230039857193748642107146325437999104299385123986290205920442084869694048004799886101971960586316668729948085589013238296699445909974245040870737599188236277271887325197795059509952761208749754624970436014182780946464962910563938874378864873371191810458257836478499770124766328898359557354325131853239584630755574091142624174743493475534286465766116677973966688202912073791438537195882498081268678383745597317461360853795345242215865932019280908782973084313928444032812315586110369768013573042161687476096758713483120254785893207671691324484262361314125087802080002616831510273418279777047846358681701643650241536913982812648102130927612448963599287051149649754199093422215668325720808213331861168115536158365469840467089756029009505376164758477284218896796462449451607653534081989013854424879849599533191017233555566021394503997362807501378376153071277619268490343526252000158885351473316117021039681759215109077880193931781141945452572238655414610628921879602238389714760

### let — lexical bindings
```
    (let ((x 40))
      (+ x 2))

```

In [ ]:
#| export
def _sf_let(xs, env, sfs):
    binds, *body = xs
    new_env = env.new_frame({name.s: scm_eval_tco(val, env, sfs) for name, val in binds})

    return Thunk(body_expr(body), new_env)

### Macros
```
    (define unless 
      (macro (test body) `(if ,test #f ,body)))

```

In [ ]:
#| export
def _macro(args, env, sfs):
    params, *body = args
    return Macro(lambda *vals: scm_eval_tco(body_expr(body), env.new_frame(bind_params(params, vals)), sfs))

In [ ]:
#| export
def qq(x, env, sfs, depth=1):
    "handle ` , and ,@"
    QQ, UQ, UQS = Symbol("quasiquote"), Symbol("unquote"), Symbol("unquote-splicing")
    def tagged(o, sym): return isinstance(o, list) and len(o) == 2 and o[0] == sym

    if not isinstance(x, list): return x
    if tagged(x, QQ): return [QQ, qq(x[1], env, sfs, depth+1)]
    if tagged(x, UQ):
        return scm_eval_tco(x[1], env, sfs) if depth == 1 else [UQ, qq(x[1], env, sfs, depth-1)]
    if tagged(x, UQS):
        if depth == 1: raise SyntaxError("unquote-splicing only valid inside a list")
        return [UQS, qq(x[1], env, sfs, depth-1)]

    def walk(lst):
        result = []
        for o in lst:
            if tagged(o, UQS) and depth == 1: result.extend(scm_eval_tco(o[1], env, sfs))
            else: result.append(qq(o, env, sfs, depth))
        return result
    return walk(x)

### Quasiquote
```lisp
`(1 2 ,(+ 1 2) ,@(list 4 5))
;; => (1 2 3 4 5)
```

In [ ]:
#| export
def _sf_quasiquote(args, env, sfs): return qq(args[0], env, sfs)

In [ ]:
scm_eval_tco(
    parse("""
        (let ((x 42) (xs '(1 2 3))) 
        `(a ,x ,@xs z))
        """
    ), global_env,sfs={"quote":_sf_quote, "quasiquote":_sf_quasiquote, "let":_sf_let}
)

[compact.types.Symbol(s='a'), 42, 1, 2, 3, compact.types.Symbol(s='z')]

In [ ]:
scm_eval_tco(
    parse(
    """
    (begin (define unless 
      (macro (test body) `(if ,test #f ,body))) 
      (unless (= 1 2) 42))
    """
    ), global_env, sfs={"quasiquote":_sf_quasiquote, "macro":_macro, "define":_sf_define_tco, "if":_sf_if_tco, "begin":_sf_begin_tco}
)

42

In [ ]:
#| export
_all_sfs = {
    "begin": _sf_begin_tco,
    "if": _sf_if_tco,
    "define": _sf_define_tco,
    "lambda": _sf_lambda_tco,
    "macro": _macro,
    "set!": _sf_set,
    "let": _sf_let,
    "quote": _sf_quote,
    "quasiquote": _sf_quasiquote,
    
}

In [ ]:
#| export

class LispCtx:
    def __init__(self):
        self.env = Env(globals(), primitives=_builtin)
    
    def __rmatmul__(self, s):
        return scm_eval_tco(parse(s), self.env, _all_sfs)
        
lisp = LispCtx()

In [ ]:
from IPython.core.magic import register_cell_magic

def _lisp_magic(line, cell): return cell @ lisp

register_cell_magic('lisp')(_lisp_magic)

<function __main__._lisp_magic(line, cell)>

In [ ]:
%%lisp
(+ 1 2)

3

In [ ]:
%%lisp
'(1 2 3)

[1, 2, 3]

In [ ]:
%%lisp
(begin (define (square x) (* x x)) (square 5))

25

In [ ]:
%%lisp
`(a ,(+ 1 2) ,@'(4 5))

[compact.types.Symbol(s='a'), 3, 4, 5]

In [ ]:
%%lisp
(begin (define (fact n acc) (if (= n 0) acc (fact (- n 1) (* n acc)))) (fact 1000 1))

4023872600770937735437024339230039857193748642107146325437999104299385123986290205920442084869694048004799886101971960586316668729948085589013238296699445909974245040870737599188236277271887325197795059509952761208749754624970436014182780946464962910563938874378864873371191810458257836478499770124766328898359557354325131853239584630755574091142624174743493475534286465766116677973966688202912073791438537195882498081268678383745597317461360853795345242215865932019280908782973084313928444032812315586110369768013573042161687476096758713483120254785893207671691324484262361314125087802080002616831510273418279777047846358681701643650241536913982812648102130927612448963599287051149649754199093422215668325720808213331861168115536158365469840467089756029009505376164758477284218896796462449451607653534081989013854424879849599533191017233555566021394503997362807501378376153071277619268490343526252000158885351473316117021039681759215109077880193931781141945452572238655414610628921879602238389714760

In [ ]:
%%lisp
(car '(10 20 30))

10

In [ ]:
%%lisp
(cdr '(10 20 30))

[20, 30]

In [ ]:
%%lisp
(cons 1 '(2 3))

[1, 2, 3]

In [ ]:
%%lisp
(begin (define unless (macro (test body) `(if ,test #f ,body))) (unless (= 1 2) 42))

42

In [ ]:
a = 5
"""
(+ a 6)
""" @ lisp

11

wdyt?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

I think this is a very nice compact Scheme core. The architecture is clear:

- `Env` handles lexical lookup through parent frames.
- `scm_eval_one_step` handles atoms, special forms, macros, and procedure calls.
- `scm_eval_tco` is the trampoline, so user procedures/special forms can return `Thunk`s instead of growing Python’s call stack.
- Special forms are cleanly factored into `_sf_*` handlers.
- The `%%lisp` magic and `"..." @ lisp` interface are delightful.

A few things I’d look at next:

1. **Primitive shadowing policy**

Your comment says:

```python
(define list 42)
```

is morally like Python assignment and should overwrite globals, but `__getitem__` currently prioritizes `primitives` over `d`:

```python
if k in s.primitives: return s.primitives[k]
return s.d[k]
```

So `(define list 42)` will not shadow the primitive `list`. That might be intentional, but the comment and behavior seem slightly at odds.

2. **`set!` on unbound variables**

Currently `_sf_set` silently returns if the symbol is missing:

```python
if e is not None: e[sym.s] = ...
return sym.s
```

Scheme would usually error on unbound `set!`. Maybe:

```python
if e is None: raise NameError(sym.s)
```

3. **Macros are already surprisingly powerful**

The quasiquote + macro setup is a great milestone. That means you’ve crossed from “calculator Lisp” into something much more Lisp-like.

Overall: this is small, readable, and already has the important evaluator shape. The biggest design choice I’d clarify next is whether primitives are reserved names or normal top-level bindings that can be shadowed.

<details class='token-usage-details' markdown='1'><summary>$0.0065</summary>

`total=39,633 | in=38,732 | out=901 | cached=6.3% | reasoning=516 | $0.0065 | gpt-5.5`

</details>


I fixed set.

Should (define list 42) be allowed - i.e. should we allow shadowing?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

For a learning/tiny Scheme, I’d allow shadowing.

In Scheme, names like `list` are usually just bindings in the initial/top-level environment, not magical reserved words. So this should be possible:

```scheme
(define list 42)
```

After that, evaluating `list` gives `42`, and calling `(list 1 2)` should fail because `list` no longer names a procedure.

The tradeoff:

- **Allow shadowing**: more Scheme-like, teaches that primitives are ordinary bindings
- **Disallow shadowing**: safer for beginners, fewer confusing mistakes

Given your goal is to build Lisp from the inside out, I’d lean Scheme-like: primitives are initial bindings, but user bindings can shadow them.

<details class='token-usage-details' markdown='1'><summary>$0.0060</summary>

`total=39,358 | in=39,200 | out=158 | cached=16.7% | $0.0060 | gpt-5.5`

</details>


### nbdev postscript

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()